In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, classification_report

In [ ]:
# ==========================================
# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# ==========================================
print("Загрузка данных MNIST (подождите около минуты)...")
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

# Нормализуем пиксели от 0 до 1
X = X / 255.0
y = y.astype(int)

# Делим на обучение (80%) и тест/экзамен (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Функция для One-Hot Encoding (правильных ответов)
def to_one_hot(y, num_classes=10):
    oh = np.zeros((y.size, num_classes))
    oh[np.arange(y.size), y] = 1
    return oh

Y_train_oh = to_one_hot(y_train)

In [ ]:
# ==========================================
# 2. КЛАСС НЕЙРОСЕТИ
# ==========================================
class SimpleNN:
    def __init__(self):
        # 784 входа -> 64 нейрона (скрытый слой) -> 10 выходов
        self.W1 = np.random.randn(784, 64)
        self.b1 = np.zeros((1, 64))
        
        self.W2 = np.random.randn(64, 10)
        self.b2 = np.zeros((1, 10))

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = np.maximum(0, self.Z1) # Активация ReLU
        
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        # Акстивация Softmax
        expZ = np.exp(self.Z2 - np.max(self.Z2, axis=1, keepdims=True))
        self.A2 = expZ / np.sum(expZ, axis=1, keepdims=True)
        return self.A2

    def backward(self, X, Y_true, lr, loss_type):
        m = X.shape[0]

        if loss_type == 'ce':
            dZ2 = self.A2 - Y_true # Идеальный градиент
        else: # 'mse'
            # ПРАВИЛЬНЫЙ ГРАДИЕНТ MSE + SOFTMAX
            diff = self.A2 - Y_true 
            dZ2 = self.A2 * diff - self.A2 * np.sum(self.A2 * diff, axis=1, keepdims=True)

        # Считаем градиенты для весов
        dW2 = (1/m) * np.dot(self.A1.T, dZ2)
        db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)
        
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * (self.Z1 > 0) # Производная ReLU
        
        dW1 = (1/m) * np.dot(X.T, dZ1)
        db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)

        # Обновляем веса
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        
    def compute_loss(self, Y_true, loss_type):
        m = Y_true.shape[0]
        if loss_type == 'ce':
            # Добавляем 1e-8 чтобы логарифм не взорвался от нуля
            return -np.mean(np.sum(Y_true * np.log(self.A2 + 1e-8), axis=1))
        else:
            return np.mean(np.sum((self.A2 - Y_true)**2, axis=1))

In [ ]:
# ==========================================
# 3. ОБУЧЕНИЕ с мини-батчами
# ==========================================
epochs = 90       # Количество эпох
batch_size = 128   # Размер пачки картинок
lr = 1.5e-2        # Скорость обучения

nn_ce = SimpleNN()
nn_mse = SimpleNN()
# Копируем веса, чтобы старт был абсолютно одинаковым
nn_mse.W1, nn_mse.W2 = nn_ce.W1.copy(), nn_ce.W2.copy()
nn_mse.b1, nn_mse.b2 = nn_ce.b1.copy(), nn_ce.b2.copy()

history = {'ce_loss': [], 'ce_acc': [], 'mse_loss': [], 'mse_acc': []}
m_train = X_train.shape[0]

print("Начинаем эксперимент...")
for i in range(epochs):
    # Перемешиваем данные перед каждой эпохой
    permutation = np.random.permutation(m_train)
    X_train_shuffled = X_train[permutation]
    Y_train_shuffled = Y_train_oh[permutation]
    y_train_shuffled = y_train[permutation] # Для расчета Accuracy
    
    # Обучаем по пачкам (Batches)
    for j in range(0, m_train, batch_size):
        X_batch = X_train_shuffled[j:j+batch_size]
        Y_batch = Y_train_shuffled[j:j+batch_size]
        
        # Шаг для CE
        nn_ce.forward(X_batch)
        nn_ce.backward(X_batch, Y_batch, lr, 'ce')
        
        # Шаг для MSE
        nn_mse.forward(X_batch)
        nn_mse.backward(X_batch, Y_batch, lr, 'mse')
        
    # --- ОЦЕНКА ПОСЛЕ КАЖДОЙ ЭПОХИ (НА ТРЕНИРОВОЧНЫХ ДАННЫХ) ---
    # Для CE
    out_ce_train = nn_ce.forward(X_train)
    loss_ce_train = nn_ce.compute_loss(Y_train_oh, 'ce')
    acc_ce_train = np.mean(np.argmax(out_ce_train, axis=1) == y_train)
    history['ce_loss'].append(loss_ce_train)
    history['ce_acc'].append(acc_ce_train)
    
    # Для MSE
    out_mse_train = nn_mse.forward(X_train)
    loss_mse_train = nn_mse.compute_loss(Y_train_oh, 'mse')
    acc_mse_train = np.mean(np.argmax(out_mse_train, axis=1) == y_train)
    history['mse_loss'].append(loss_mse_train)
    history['mse_acc'].append(acc_mse_train)
    
    print(f"Эпоха {i+1}/{epochs} (Train) | CE: Acc {acc_ce_train:.2%}, Loss {loss_ce_train:.4f} | MSE: Acc {acc_mse_train:.2%}, Loss {loss_mse_train:.4f}")

# --- ФИНАЛЬНЫЙ ЭКЗАМЕН (ОДИН РАЗ НА ТЕСТОВЫХ ДАННЫХ) ---
print("\n--- ФИНАЛЬНАЯ ПРОВЕРКА НА ТЕСТОВЫХ ДАННЫХ ---")

# Оценка CE
out_ce_test = nn_ce.forward(X_test)
acc_ce_test = np.mean(np.argmax(out_ce_test, axis=1) == y_test)
loss_ce_test = nn_ce.compute_loss(to_one_hot(y_test), 'ce')

# Оценка MSE
out_mse_test = nn_mse.forward(X_test)
acc_mse_test = np.mean(np.argmax(out_mse_test, axis=1) == y_test)
loss_mse_test = nn_mse.compute_loss(to_one_hot(y_test), 'mse')

print(f"Результат на ТЕСТЕ | Cross-Entropy: Acc {acc_ce_test:.2%}, Loss {loss_ce_test:.4f}")
print(f"Результат на ТЕСТЕ | MSE:           Acc {acc_mse_test:.2%}, Loss {loss_mse_test:.4f}")

In [ ]:
# ==========================================
# 4. РИСУЕМ КРАСИВЫЕ ГРАФИКИ
# ==========================================
plt.figure(figsize=(14, 5))

# График Loss (Ошибки)
plt.subplot(1, 2, 1)
plt.plot(history['ce_loss'], label='Cross-Entropy', color='blue', linewidth=2)
plt.plot(history['mse_loss'], label='MSE', color='red', linewidth=2)
plt.title('График падения функции потерь (Loss)')
plt.xlabel('Эпоха')
plt.ylabel('Значение ошибки')
plt.grid(True)
plt.legend()

# График Accuracy (Точности)
plt.subplot(1, 2, 2)
plt.plot(history['ce_acc'], label='Cross-Entropy', color='blue', linewidth=2)
plt.plot(history['mse_acc'], label='MSE', color='red', linewidth=2)
plt.title('Точность на тестовой выборке (Accuracy)')
plt.xlabel('Эпоха')
plt.ylabel('Доля верных ответов')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 5. МАТРИЦЫ ОШИБОК И ФИНАЛЬНЫЙ ОТЧЕТ
# ==========================================
pred_ce = np.argmax(nn_ce.forward(X_test), axis=1)
pred_mse = np.argmax(nn_mse.forward(X_test), axis=1)

f1_ce = f1_score(y_test, pred_ce, average='macro')
f1_mse = f1_score(y_test, pred_mse, average='macro')

print("\n=== ИТОГОВЫЕ МЕТРИКИ КАЧЕСТВА ===")
# Вместо history['ce_acc'][-1] используем наши переменные из блока "ФИНАЛЬНЫЙ ЭКЗАМЕН"
print(f"Cross-Entropy -> Test Accuracy: {acc_ce_test*100:.2f}%, Macro F1-Score: {f1_ce:.4f}")
print(f"MSE           -> Test Accuracy: {acc_mse_test*100:.2f}%, Macro F1-Score: {f1_mse:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm_ce = confusion_matrix(y_test, pred_ce)
sns.heatmap(cm_ce, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title(f'Матрица ошибок: Cross-Entropy (F1: {f1_ce:.4f})')
axes[0].set_xlabel('Предсказано сетью')
axes[0].set_ylabel('Истинная цифра')

cm_mse = confusion_matrix(y_test, pred_mse)
sns.heatmap(cm_mse, annot=True, fmt='d', cmap='Reds', ax=axes[1], cbar=False)
axes[1].set_title(f'Матрица ошибок: MSE (F1: {f1_mse:.4f})')
axes[1].set_xlabel('Предсказано сетью')
axes[1].set_ylabel('Истинная цифра')

plt.tight_layout()
plt.show()